# TTSPort Testing Guide

This notebook tests the TTSPort implementation with real API calls.

**Prerequisites:**
- Set `ELEVENLABS_API_KEY` environment variable for ElevenLabs tests
- Set `OPENAI_API_KEY` environment variable for OpenAI tests

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv
load_dotenv()

print(f"ELEVENLABS_API_KEY set: {bool(os.getenv('ELEVENLABS_API_KEY'))}")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")

## 1. ElevenLabs TTS

In [ ]:
from chatforge.adapters.tts import ElevenLabsTTSAdapter, ElevenLabsVoiceConfig
from chatforge.ports.tts import AudioFormat, AudioQuality

In [ ]:
# List available voices
async with ElevenLabsTTSAdapter() as tts:
    voices = await tts.list_voices()
    print(f"Found {len(voices)} voices:\n")
    for v in voices[:5]:  # Show first 5
        print(f"  - {v.name} ({v.voice_id})")

In [ ]:
# Synthesize with ElevenLabs
async with ElevenLabsTTSAdapter() as tts:
    # Get first available voice
    voices = await tts.list_voices()
    voice_id = voices[0].voice_id if voices else "21m00Tcm4TlvDq8ikWAM"
    
    config = ElevenLabsVoiceConfig(
        voice_id=voice_id,
        stability=0.5,
        similarity_boost=0.75,
    )
    
    result = await tts.synthesize(
        "Hello world! This is a test of the ElevenLabs text to speech adapter.",
        config,
        output_format=AudioFormat.MP3,
        quality=AudioQuality.HIGH,
    )
    
    print(f"Audio bytes: {len(result.audio_bytes)}")
    print(f"Format: {result.format}")
    print(f"Sample rate: {result.sample_rate}")
    print(f"Input characters: {result.input_characters}")
    
    # Save to file
    with open("elevenlabs_output.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("\nSaved to elevenlabs_output.mp3")

## 2. OpenAI TTS

In [ ]:
from chatforge.adapters.tts import OpenAITTSAdapter, OpenAIVoiceConfig
from chatforge.ports.tts import AudioFormat, AudioQuality

In [ ]:
# List available voices
async with OpenAITTSAdapter() as tts:
    voices = await tts.list_voices()
    print(f"Found {len(voices)} voices:\n")
    for v in voices:
        print(f"  - {v.name} ({v.voice_id})")

In [ ]:
# Synthesize with OpenAI
async with OpenAITTSAdapter() as tts:
    config = OpenAIVoiceConfig(
        voice_id="nova",
        style_prompt="Speak in a warm, friendly tone",
    )
    
    result = await tts.synthesize(
        "Hello world! This is a test of the OpenAI text to speech adapter.",
        config,
        output_format=AudioFormat.MP3,
        quality=AudioQuality.HIGH,
    )
    
    print(f"Audio bytes: {len(result.audio_bytes)}")
    print(f"Format: {result.format}")
    print(f"Sample rate: {result.sample_rate}")
    print(f"Input characters: {result.input_characters}")
    
    # Save to file
    with open("openai_output.mp3", "wb") as f:
        f.write(result.audio_bytes)
    print("\nSaved to openai_output.mp3")

## 3. Streaming Example

In [ ]:
# Streaming with OpenAI
async with OpenAITTSAdapter() as tts:
    config = OpenAIVoiceConfig(voice_id="alloy")
    
    chunks = []
    async for chunk in tts.stream(
        "This is a streaming test. The audio is generated in chunks.",
        config,
    ):
        chunks.append(chunk)
        print(f"Received chunk: {len(chunk)} bytes")
    
    total_bytes = sum(len(c) for c in chunks)
    print(f"\nTotal: {len(chunks)} chunks, {total_bytes} bytes")
    
    # Save streamed audio
    with open("openai_streamed.mp3", "wb") as f:
        for chunk in chunks:
            f.write(chunk)
    print("Saved to openai_streamed.mp3")

## 4. Play Audio (Optional)

If you have IPython.display available, you can play the audio directly:

In [ ]:
from IPython.display import Audio, display

# Play the OpenAI output
display(Audio("openai_output.mp3"))